In [3]:
import os
import smtplib
from email.mime.text import MIMEText
from email.mime.multipart import MIMEMultipart
from dotenv import load_dotenv
import markdown
import textwrap

load_dotenv(override=True)

def send_market_report_email(receiver_email: str, subject: str, markdown_content: str):
    smtp_server = os.getenv("SMTP_SERVER", "smtp.qq.com")
    smtp_port = int(os.getenv("SMTP_PORT", 465))
    sender_email = os.getenv("SENDER_EMAIL")
    sender_password = os.getenv("SENDER_PASSWORD")

    if not sender_email or not sender_password:
        print("--- 警告: 未配置发件人邮箱凭证，跳过邮件发送 ---")
        return False

    try:
        # 1. 将 Markdown 转为 HTML
        cleaned_markdown = textwrap.dedent(markdown_content).strip()
        html_body = markdown.markdown(cleaned_markdown)

        # 加上一点基础的 CSS 样式，让邮件更好看
        html_content = f"""
        <html>
        <head>
            <style>
                body {{ font-family: Arial, sans-serif; line-height: 1.6; color: #333; }}
                h1 {{ color: #2c3e50; border-bottom: 2px solid #eee; padding-bottom: 5px; }}
                h2 {{ color: #34495e; margin-top: 20px; }}
                ul {{ padding-left: 20px; }}
            </style>
        </head>
        <body>
            {html_body}
        </body>
        </html>
        """

        # 2. 构建邮件
        message = MIMEMultipart("alternative")
        message["Subject"] = subject
        message["From"] = sender_email
        message["To"] = receiver_email

        # 必须显式指定 "html"
        html_part = MIMEText(html_content, "html", "utf-8")
        message.attach(html_part)

        # 3. 发送
        with smtplib.SMTP_SSL(smtp_server, smtp_port) as server:
            server.login(sender_email, sender_password)
            server.sendmail(sender_email, receiver_email, message.as_string())

        print(f"--- 成功发送邮件至 {receiver_email} ---")
        return True
    except Exception as e:
        print(f"--- 发送邮件失败: {e} ---")
        return False

In [4]:
if __name__ == "__main__":
    # 测试用的 Markdown 报告内容
    test_markdown = """
    # 市场情报测试报告

    ## 1. 核心发现
    - 这是一个由 **AI Agent** 自动生成的测试报告。
    - 验证 Markdown 到 HTML 的转换与邮件发送功能。

    ## 2. 行动建议
    - 确认收件箱是否正常收到排版后的富文本邮件。
    """

    # 替换成你自己的接收邮箱（可以是同一个邮箱或者另一个测试邮箱）
    receiver = "zhukunh@student.unimelb.edu.au"
    subject = "【测试】市场智能分析报告"

    print("--- 正在发送测试邮件 ---")
    success = send_market_report_email(receiver, subject, test_markdown)
    if success:
        print("测试成功！请检查你的邮箱。")
    else:
        print("测试失败，请检查配置。")

--- 正在发送测试邮件 ---
--- 成功发送邮件至 zhukunh@student.unimelb.edu.au ---
测试成功！请检查你的邮箱。
